# Practical Homework 2: Support Vector Machines

ML 2

Amanda Boschman

April 2026

## 1. Introduction and Project Goal

### 1.1 Assignment Goal:

Goal: Accurately predict whether or not a person has heart disease. Using selected predictors, an SVM model will be used to determine a decision boundary (linear, radial, or polynomial) in terms of classifying the data.


### 1.2 Research Question

Can demographic and health behavior be sufficient enough to classify whether or not a respondent has heart disease?

## 2. Set Up

### 2.1 Import Packages

In [1]:
#Imports
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, precision_score, recall_score, f1_score
from sklearn.svm import SVC
from sklearn.inspection import DecisionBoundaryDisplay

### 2.2 Load in the Data

In [2]:
nhis = pd.read_csv('../data/nhis.csv')

### 2.3 Inspect the Data

In [ ]:
nhis.head()

In [ ]:
nhis.shape

In [ ]:
nhis.columns

## 3. Variable Selection

### 3.1 Disease Target Candidates

In [6]:
diseases = ['CANCEREV', 'CHEARTDIEV', 'DIABETICEV', 'HEARTATTEV', 'STROKEV']

In [ ]:
nhis[diseases].head()

### 3.2 Selected Target Variable

In [ ]:
target = 'CHEARTDIEV'
print('Selected Target Variable:', target)

### 3.3 Predictor Candidates

In [ ]:
all_predictors = nhis.drop(columns=diseases).columns
print("Predictor Candidates:", all_predictors)

### 3.4 Selected Predictors

1. 'SEX'
2. 'MOD10DMIN'
3. 'HRSLEEP'
4. 'BMICALC'
5. 'VEGENO'
6. 'ALCDAYSYR'

In [10]:
predictors = ['SEX', 'MOD10DMIN', 'HRSLEEP', 'BMICALC', 'VEGENO', 'ALCDAYSYR']

In [ ]:
nhis[predictors].head()

### 3.5 Notes from the Codebook
Selected Target (Disease):
1. 'CHEARTDIEV': if the respondent has coronary heart disease (0 = Not in Universe, 1 = No, 2 = Yes, 7 or 8 or 9 = Unknown)

Selected Predictors:
1. 'SEX': 1 = Male, 2 = Female
2. 'MOD10DMIN': Duration of moderate leisure-time physical activities (000 = Not in Universe)
3. 'HRSLEEP': Average number of hours of sleep per day (01 through 24, 00 = Not in Universe, 25 = less than 1 hour)
4. 'BMICALC': Body Mass Index (BMI) (a 4 digit numeric variable with one implied decimal place...so 0123 is 12.3) (0.0 = Not in Universe, 996.0 = Not calculable)
5. 'VEGENO': how many times a respondent ate vegetables in a specified time period (0 = never or less than 6 times per year, 995 = 995+ times, 996 = Not in Universe, 997 or 998 or 999 = Unknown)
6. 'ALCDAYSYR': number of days per year during the past year the respondent drank alcholic beverages (000 to 365) (995 = Inconsistent, 996 = Not in Universe, 997 or 998 or 999 = Unknown)

## 4. Data Cleaning

### 4.1 Check Missing Values and Invalid Codes

In [ ]:
#Check predictors for NaNs
nhis[predictors].isna().sum()

In [ ]:
#Check target for NaNs
nhis[target].isna().sum()

In [ ]:
#Check predictors for invalid values
for col in predictors:
    print(f'\n{col}')
    print(sorted(nhis[col].unique().tolist()))

In [ ]:
#Check target for invalid values
nhis[target].unique().tolist()

Although there are not any NaNs, there are invalid codes.

Predictors:
1. 'SEX': 7 and 9 indicate 'Unknown'
2. 'MOD10DMIN': 0 indicates 'NA'; 996 indicates 'Error'; 997, 998, and 999 indicate 'Unknown'
3. 'HRSLEEP': 0 indicates 'NA' or 'Not asked'; 97, 98, and 99 indicate 'Unknown'
4. 'BMICALC': 996 indicates 'Not Calculable'
5. 'VEGENO': 996 indicates 'Not in Universe'; 997, 998, and 999 indicates 'Unknown'
6. 'ALCDAYSYR': 996 indicate 'Not in Universe'; 997, 998, and 999 indicate 'Unknown'

Target:
1. 'CHEARTDIEV': 0 indicates 'Not in Universe'; 7, 8, and 9 indicate 'Unknown'

### 4.2 Clean Rows Based on Target Variable

In [ ]:
print('Rows with invalid CHEARTDIEV values:', int(nhis[target].isin([0, 7, 8, 9]).sum()))

print('Rows with valid CHEARTDIEV values:', int(nhis[target].isin([1, 2]).sum()))

First, we must drop the rows that do not have a valid value for the target variable.

In [ ]:
#Drop rows with invalid values for the target variable
nhis_heart = nhis[nhis[target].isin([1, 2])].copy()
nhis_heart.shape

In [18]:
#Reindex since rows were dropped
nhis_heart = nhis_heart.reset_index(drop=True)

### 4.3 Recode the Target Variable

Currently, the target:
1. 'CHEARTDIEV': 1 = No, 2 = Yes

Recode the target variable to be compatible with SVMs:
1. 'CHEARTDIEV': 0 = No, 1 = Yes

In [19]:
#Recode target
nhis_heart[target] = nhis_heart[target].map({
    1: 0,
    2: 1
})

In [ ]:
#Checking recoding
nhis_heart[target].value_counts(dropna=False).sort_index()

### 4.4 Clean Predictors

Final Predictors Code Notes:
1. 'SEX': 1 = Male, 2 = Female
2. 'MOD10DMIN': duration of moderate leisure-time physical activities
3. 'HRSLEEP': average number of hours of sleep per day (1 to 24)
4. 'BMICALC': Body Mass Index (BMI) (a 4 digit number ***FILL IN)
5. 'VEGENO': how many times a respondent ate vegetables in a specified time period
6. 'ALCDAYSYR': number of days in the past year the respondent drank alcholic beverages (0 to 365)

Predictors:
1. 'SEX': 7 and 9 indicate 'Unknown'
2. 'MOD10DMIN': 0 indicates 'NA'; 996 indicates 'Error';997, 998, and 999 indicate 'Unknown'
3. 'HRSLEEP': 0 indicates 'NA' or 'Not asked'; 97, 98, and 99 indicate 'Unknown'
4. 'BMICALC': 996 indicates 'Not Calculable'
5. 'VEGENO': 996 indicates 'Not in Universe'; 997, 998, and 999 indicates 'Unknown'
6. 'ALCDAYSYR': 996 indicate 'Not in Universe'; 997, 998, and 999 indicate 'Unknown'

In [ ]:
#Invalid predictors
invalid_predictors = pd.DataFrame({
    'SEX': nhis_heart['SEX'].isin([7, 9]),
    'MOD10DMIN': nhis_heart['MOD10DMIN'].isin([0, 996, 997, 998, 999]),
    'HRSLEEP': nhis_heart['HRSLEEP'].isin([0, 97, 98, 99]),
    'BMICALC': nhis_heart['BMICALC'].isin([996]),
    'VEGENO': nhis_heart['VEGENO'].isin([996, 997, 998, 999]),
    'ALCDAYSYR': nhis_heart['ALCDAYSYR'].isin([996, 997, 998, 999])
})

#Count of invalid predictors per row
invalid_count_per_row = invalid_predictors.sum(axis=1)

#Count of counts of invalid predictors per row
#E.g. Number of rows with 1 invalid predictor
print('Number of rows with 0, 1, 2, 3, 4, 5 invalid predictors:')
print(invalid_count_per_row.value_counts().sort_index())

In [ ]:
#Shape of the Data
nhis_heart.shape

In [23]:
nhis_heart_clean = nhis_heart[invalid_count_per_row == 0].copy()

In [ ]:
#Shape of data
nhis_heart_clean.shape

In [ ]:
#Invalid predictors
invalid_predictors = pd.DataFrame({
    'SEX': nhis_heart_clean['SEX'].isin([7, 9]),
    'MOD10DMIN': nhis_heart_clean['MOD10DMIN'].isin([0, 997, 998, 999]),
    'HRSLEEP': nhis_heart_clean['HRSLEEP'].isin([0, 97, 98, 99]),
    'BMICALC': nhis_heart_clean['BMICALC'].isin([996]),
    'VEGENO': nhis_heart_clean['VEGENO'].isin([996, 997, 998, 999]),
    'ALCDAYSYR': nhis_heart_clean['ALCDAYSYR'].isin([996, 997, 998, 999])
})

#Count of invalid predictors per row
invalid_count_per_row_clean = invalid_predictors.sum(axis=1)

#Count of counts of invalid predictors per row
#E.g. Number of rows with 1 invalid predictor
print('Number of rows with 0, 1, 2, 3, 4, 5 invalid predictors:')
print(invalid_count_per_row_clean.value_counts().sort_index())

Now, the dataset has 0 invalid predictor values.

In [ ]:
#Check unique values in each column for any invalid codes
for col in predictors:
    print(f'\n{col}')
    print(sorted(nhis_heart_clean[col].unique().tolist()))

print(sorted(nhis_heart_clean[target].unique().tolist()))

In [27]:
#Reindex the cleaned data
nhis_heart_clean = nhis_heart_clean.reset_index(drop=True)

All remaining rows have all valid predictor values.

### 4.5 Recode Predictor Variables

All predictor variables, except SEX, are numerical. So, only SEX must be one hot encoded.

In [28]:
#One Hot Encode predictor 'SEX'
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
encoded_sex = pd.DataFrame(encoder.fit_transform(nhis_heart_clean[['SEX']]),
                                                 columns=encoder.get_feature_names_out(['SEX']),
                                                 index=nhis_heart_clean.index)

#Drop old column
nhis_heart_clean = nhis_heart_clean.drop(columns=['SEX'])

#Add new encoded column
nhis_heart_clean = pd.concat([nhis_heart_clean, encoded_sex], axis=1)

In [ ]:
nhis_heart_clean.columns

One Hot Encoded leads to two variables: SEX_1, SEX_2


In [30]:
predictors = [col for col in predictors if col != 'SEX'] + ['SEX_1', 'SEX_2']

### 4.6 Subset to Target and Predictor Variables

In [31]:
X = nhis_heart_clean[predictors]

y = nhis_heart_clean[target]

### 4.7 Final Cleaned Dataset Inspection

In [ ]:
X.shape

In [ ]:
X.head()

In [ ]:
y.shape

In [ ]:
y.head()

Final Predictors Code Notes:
1. 'SEX': 0 = Male, 1 = Female
2. 'MOD10DMIN': duration of moderate leisure-time physical activities
3. 'HRSLEEP': average number of hours of sleep per day (1 to 24)
4. 'BMICALC': Body Mass Index (BMI)
5. 'VEGENO': how many times a respondent ate vegetables in a specified time period
6. 'ALCDAYSYR': number of days in the past year the respondent drank alcholic beverages (0 to 365)

Final Target Code Notes:
1. 'CHEARTDIEV': if the respondent has coronary heat disease (0 = does not have heart disease, 1 = has heart disease)

## 5. Exploratory Data Analysis

This section explores the cleaned modeling dataset before fitting the SVM models. The target class balance is checked, the predictor variables are summarized, and basic patterns between the predictors and the target variable (heart disease) are explored. This is all to check for class imbalance, strange predictor distributions, and any relationships that may affect the models later made.

### 5.1 Target Class Balance

This section checks to see if there is an approximately even division of values in the target variable 'CHEARTDIEV'.

In [ ]:
#Check for class imbalance in target variable
print('Target Variable Value Counts:', y.value_counts())

print('Target Variable Percentages:', (y.value_counts(normalize=True).sort_index() * 100).round(2))

There is severe class imbalance presence with about 95.22% of the respondents not having heart disease and about 4.78% of the respondents having heart disease.

For evaluation metrics, accuracy alone will not be reliable. Precision, recall, F1-score, and the evaluation matrix will be more valuable and reliable.

Also, the train test split needs to use straify=y so both sets preserve the class imbalance.

### 5.2 Summary Statistics for Numeric Predictors

This section summarizes the numeric predictor variables after cleaning. Summary statistics include center, spread, minimum, and maximum values.

In [ ]:
numeric_predictors = [col for col in predictors 
                      if col not in ['SEX_1', 'SEX_2']]

print('Numeric Predictors:', numeric_predictors)

In [ ]:
nhis_heart_clean[numeric_predictors].describe().T

The numeric predictors are in valid ranges. Looking at the minimum and maximum values, the scale variation between predictors is visible. 'MOD10DMIN', 'VEGENO', and 'ALCDAYSYR' have maximum values of 720.0, 230.0, and 365.0. However, 'HRSLEEP' and 'BMICALC' have maximum values of 22.0 and 52.3. So, 'MOD10DMIN', 'VEGENO', and 'ALCDAYSYR' have a larger scale than 'HRSLEEP' and 'BMICALC'.

SVMs are sensitive to the scale of predictor variables; therefore, StandardScaler will be used to address the variations in scaling of the predictor variables.

There also appears to be a right skew in the 'MOD10DMIN', 'VEGENO', and 'ALCDAYSYR' variables.

### 5.3 Counts for Categorical Predictor

This section checks the distribution of the 'SEX' variable that was one hot encoded to be two variables: 'SEX_1' and 'SEX_2'.

In [ ]:
sex_encoded_cols = ['SEX_1', 'SEX_2']

sex_encoded_counts = nhis_heart_clean[sex_encoded_cols].sum()

print('Counts:')
print('Male:', sex_encoded_counts[0])
print('Female:', sex_encoded_counts[1])

print('----------------')

print('Percentages:')
print('Male:', ((sex_encoded_counts[0]/len(nhis_heart_clean))*100).round(2))
print('Female:', ((sex_encoded_counts[1]/len(nhis_heart_clean))*100).round(2))

Approximately 48.06% of respondents in our cleaned dataset are male and 51.94% of respondents are female. This is roughly an even split, which indicates a fairly balanced class and that both sexes are represented in the dataset.

### 5.4 Basic Plots of Target vs. Predictors

This section compares the predictors to the target variable visually before modeling.

In [ ]:
final_cols = predictors + [target]
print(final_cols)

nhis_model = nhis_heart_clean[final_cols].copy()

In [ ]:
#Boxplot of BMI by heart disease status
plt.figure(figsize=(6,4))

nhis_model.boxplot(column='BMICALC', by='CHEARTDIEV')

plt.title('BMI by Heart Disease Class')
plt.suptitle('')
plt.xlabel('Heart Disease Class')
plt.ylabel('BMI')
plt.xticks([1,2], ['No Heart Disease', 'Heart Disease'])

plt.show()

In [ ]:
#Boxplot of alcohol days per year by heart disease class
plt.figure(figsize=(6, 4))

nhis_model.boxplot(column='ALCDAYSYR', by='CHEARTDIEV')

plt.title('Alcohol Drinking Days Per Year by Heart Disease Class')
plt.suptitle('')
plt.xlabel('Heart Disease Class')
plt.ylabel('Alcohol Days Per Year')
plt.xticks([1,2], ['No Heart Disease', 'Heart Disease'])

plt.show()

The above boxplots compare 'BMICALC' and 'ALCDAYSYR' to the heart disease classification.

Visually, the median of BMI appears similar between the respondents who do not have heart disease and the respondents that do have heart disease.

Visually, the median of 'ALCDAYSYR' appears similar between the respondents who do not have heart disease and the respondents that do have heart disease.

Because the amount of respondents without heart disease is significantly larger than the amount of respondents with heart disease, the visual interpretation must be taken with caution.

### 5.5 Initial Observations

Overall, the dataset is ready for modeling, but class imbalance and predictor scaling are needed.

## 6. Modeling Preparation

### 6.1 Define X and y

In [ ]:
#predictor data
X.head()

In [ ]:
X.shape

In [ ]:
#target data
y.head()

In [ ]:
y.shape

### 6.2 Train/Test Split

In [47]:
X_train, X_test, y_train, y_test = train_test_split(X, y,
                                                    train_size=0.75,
                                                    random_state=0,
                                                    stratify=y
)

### 6.3 Scale Predictors

In [48]:
#Create scaler
scaler = StandardScaler()

#Fit scaler on training predictors
X_train_scaled = scaler.fit_transform(X_train)

#Transform testing predictors
X_test_scaled = scaler.transform(X_test)

In [49]:
#Currently scaled arrays, so convert back to DataFrames
X_train_scaled = pd.DataFrame(X_train_scaled,
                              columns=X_train.columns,
                              index=X_train.index)

X_test_scaled = pd.DataFrame(X_test_scaled,
                             columns=X_test.columns,
                             index=X_test.index)

### 6.4 Evaluation Metrics

Because the target variable is very imbalanced, evaluation metrics of accuracy, the confusion matric, precision, recall, and F1-score will be used, instead of just relying on accuracy.

## 7. Linear Support Vector Machine

### 7.1 Fit Initial Linear SVM

In [ ]:
#Initial Linear SVM
svm_linear = SVC(C=0.01, kernel='linear')
svm_linear.fit(X_train_scaled, y_train)

In [51]:
#Predictions
y_train_pred_linear = svm_linear.predict(X_train_scaled)
y_test_pred_linear = svm_linear.predict(X_test_scaled)

### 7.2 Tune C Using Cross-Validation

In [ ]:
#K-Fold Cross-Validation

kfold = KFold(5, random_state=0,shuffle=True)

grid_linear = GridSearchCV(svm_linear,
                        {'C':[0.01,0.1,1]},
                        refit=True,
                        cv=kfold,
                        scoring='f1',
                        verbose=0
)
grid_linear.fit(X_train_scaled, y_train)
best_C_linear = grid_linear.best_params_['C']
print('Best C: ', best_C_linear)

### 7.3 Evaluation Metrics on Best Linear SVM

The initial linear SVM using a C value of 0.01 is determined to be the best C value, therefore the best Linear SVM model.

In [ ]:
#Accuracy
train_accuracy_linear = accuracy_score(y_train, y_train_pred_linear)
test_accuracy_linear = accuracy_score(y_test, y_test_pred_linear)

#Confusion matrix for test set
conf_matrix_linear = confusion_matrix(y_test, y_test_pred_linear)

#Classification report for test set
class_report_linear = classification_report(y_test, y_test_pred_linear, zero_division=0)

#Print metrics
print('Linear SVM:')

print('-------')

print('Training Accuracy:', round(train_accuracy_linear, 4))
print('Testing Accuracy:', round(test_accuracy_linear, 4))

print('-------')

print('Confusion Matrix', conf_matrix_linear)

print('-------')

print('Classification Report for Test Set:', class_report_linear)

In [ ]:
#For positive heart disease class
#Precision
precision_linear = precision_score(y_test, y_test_pred_linear,
                                   pos_label=1,
                                   zero_division=0)

#Recall
recall_linear = recall_score(y_test, y_test_pred_linear,
                             pos_label=1,
                             zero_division=0)

#F1 Score
f1_linear = f1_score(y_test, y_test_pred_linear,
                     pos_label=1,
                     zero_division=0)

#Print metrics
print('Linear SVM:')

print('------')

print('Precision Score:', round(precision_linear, 4))

print('Recall:', round(recall_linear, 4))

print('F1 Score:', round(f1_linear, 4))

Using cross-validation, the best C was 0.01, which was the initial linear SVM model's C value. This smaller C value means the model favors a simpler classification boundary.

Looking at the evaluation metrics, this linear SVM model performed poorly for the class of respondents that do have heart disease. The training and testing accuracy rates were misleading due to the severe class imbalance in the data. The confusion matrix shows that every test observation was predicted to be class 0 (no heart disease). So, the model did not identify any respondents as having heart disease. Therefore, the precision score, recall, and f1 score resulted in 0.0.

## 8. Radial Support Vector Machine

### 8.1 Fit Initial Radial SVM

In [ ]:
#SVM with a radial kernel
svm_radial = SVC(C=0.01, kernel='rbf')
svm_radial.fit(X_train_scaled, y_train)

In [56]:
#Predictions
y_train_pred_radial = svm_radial.predict(X_train_scaled)
y_test_pred_radial = svm_radial.predict(X_test_scaled)

### 8.2 Tune C and Gamma using Cross-Validation

In [ ]:
#Cross validation to select an optimal cost
grid_radial = GridSearchCV(svm_radial,
                        {'C':[0.01,0.1,1], 
                        'gamma':[0.001,0.01,0.1,1,'scale']},
                        refit=True,
                        cv=kfold,
                        scoring='f1'
)
grid_radial.fit(X_train_scaled, y_train)
best_C_radial = grid_radial.best_params_['C']
best_gamma_radial = grid_radial.best_params_['gamma']
print('Best C: ', best_C_radial)
print('Best Gamma:', best_gamma_radial)

### 8.3 Evaluation Metrics on Best Radial SVM

The initial radial SVM used a C value of 0.01. Using cross validation, the best C value was found to be the same (C = 0.01). The initial radial SVM used the default 'scale' gamma value, while the best radial SVM found the best gamma value to be 0.001.

In [58]:
#Best Radial SVM
best_svm_radial = SVC(C=best_C_radial, gamma=best_gamma_radial, kernel='rbf')
best_svm_radial.fit(X_train_scaled, y_train)

#Best Radial SVM Predictions
best_y_train_pred_radial = best_svm_radial.predict(X_train_scaled)
best_y_test_pred_radial = best_svm_radial.predict(X_test_scaled)

In [ ]:
#Evaluation Metrics
#Accuracy
train_accuracy_radial = accuracy_score(y_train, best_y_train_pred_radial)
test_accuracy_radial = accuracy_score(y_test, best_y_test_pred_radial)

#Confusion matrix for test set
conf_matrix_radial = confusion_matrix(y_test, best_y_test_pred_radial)

#Classification report for test set
class_report_radial = classification_report(y_test, best_y_test_pred_radial, zero_division=0)

#Print metrics
print('Radial SVM:')

print('-------')

print('Training Accuracy:', round(train_accuracy_radial, 4))
print('Testing Accuracy:', round(test_accuracy_radial, 4))

print('-------')

print('Confusion Matrix', conf_matrix_radial)

print('-------')

print('Classification Report for Test Set:', class_report_radial)

In [ ]:
#More Evaluation Metrics
#For positive heart disease class
#Precision
precision_radial = precision_score(y_test, best_y_test_pred_radial,
                                   pos_label=1,
                                   zero_division=0)

#Recall
recall_radial = recall_score(y_test, best_y_test_pred_radial,
                             pos_label=1,
                             zero_division=0)

#F1 Score
f1_radial = f1_score(y_test, best_y_test_pred_radial,
                     pos_label=1,
                     zero_division=0)

#Print metrics
print('Radial SVM:')

print('------')

print('Precision Score:', round(precision_radial, 4))

print('Recall:', round(recall_radial, 4))

print('F1 Score:', round(f1_radial, 4))

The best radial SVM is behaving similarly to the linear SVM: misleading high accuracy rates because the target variable is very imbalanced, the model predicted every observation as class 0 indicating no heart disease (this is visible in the confusion matrix).

Since the best radial SVM did not predict any positive heart disease cases, the precision, recall, and F1-score resulted in 0.0. Thus, the best radial SVM model failed to detect heart disease.

## 9. Polynomial Support Vector Machine

### 9.1 Fit Initial Polynomial SVM

In [ ]:
#Support vector machine with a polynomial kernel
svm_poly = SVC(C=0.01, kernel='poly', degree=2)
svm_poly.fit(X_train_scaled, y_train)

### 9.2 Tune C, Degree, and Gamma Using Cross-Validation

In [ ]:
#Cross validation to select an optimal cost
grid_poly = GridSearchCV(svm_poly,
                        {'C':[0.01,0.1,1],
                         'gamma':['scale', 0.01],
                         'degree':[2,3]},
                        refit=True,
                        cv=kfold,
                        scoring='f1'
)
grid_poly.fit(X_train_scaled, y_train)
best_C_poly = grid_poly.best_params_['C']
best_gamma_poly = grid_poly.best_params_['gamma']
best_degree_poly = grid_poly.best_params_['degree']
print('Best C: ', best_C_poly)
print('Best Gamma:', best_gamma_poly)
print('Best Degree:', best_degree_poly)

### 9.3 Evaluation Metrics on Best Polynomial SVM

In [63]:
#Best Radial SVM
best_svm_poly = SVC(C=best_C_poly, gamma=best_gamma_poly, degree=best_degree_poly, kernel='poly')
best_svm_poly.fit(X_train_scaled, y_train)

#Best Radial SVM Predictions
best_y_train_pred_poly = best_svm_poly.predict(X_train_scaled)
best_y_test_pred_poly = best_svm_poly.predict(X_test_scaled)

In [ ]:
#Evaluation Metrics
#Accuracy
train_accuracy_poly = accuracy_score(y_train, best_y_train_pred_poly)
test_accuracy_poly = accuracy_score(y_test, best_y_test_pred_poly)

#Confusion matrix for test set
conf_matrix_poly = confusion_matrix(y_test, best_y_test_pred_poly)

#Classification report for test set
class_report_poly = classification_report(y_test, best_y_test_pred_poly, zero_division=0)

#Print metrics
print('Polynomial SVM:')

print('-------')

print('Training Accuracy:', round(train_accuracy_poly, 4))
print('Testing Accuracy:', round(test_accuracy_poly, 4))

print('-------')

print('Confusion Matrix', conf_matrix_poly)

print('-------')

print('Classification Report for Test Set:', class_report_poly)

In [ ]:
#More Evaluation Metrics
#For positive heart disease class
#Precision
precision_poly = precision_score(y_test, best_y_test_pred_poly,
                                   pos_label=1,
                                   zero_division=0)

#Recall
recall_poly = recall_score(y_test, best_y_test_pred_poly,
                             pos_label=1,
                             zero_division=0)

#F1 Score
f1_poly = f1_score(y_test, best_y_test_pred_poly,
                     pos_label=1,
                     zero_division=0)

#Print metrics
print('Polynomial SVM:')

print('------')

print('Precision Score:', round(precision_poly, 4))

print('Recall:', round(recall_poly, 4))

print('F1 Score:', round(f1_poly, 4))

The best polynomial SVM is behaving similarly to the other SVM models: misleading high accuracy rates because the target variable is very imbalanced, the model predicted every observation as class 0 indicating no heart disease (this is visible in the confusion matrix).

Since the best polynomial SVM did not predict any positive heart disease cases, the precision, recall, and F1-score resulted in 0.0. Thus, the best polynomial SVM model failed to detect heart disease too.

## 10. Model Comparison

All three SVM models (linear, radial, and polynomial) had similar results of high overall accuracy at around 95%, but that metric was misleading because the target variable was highly imbalanced. About 95% of the observations in the dataset were in the no heart disease class. So, by predicting every observation (or nearly every) as class 0, a model can result in high accuracy.

All three confusion matrices showed that the models did not predict any heart disease cases. The metrics of F1 score, recall, and precision for the positive heart disease class were reported to be 0.0 in all three models.

With the goal being to classify cases of positive heart disease based on the selected predictors, none of the three models are worthwhile as is. The severe class imbalance of the dataset greatly affected model performance.

## 11. SVM Decision Boundary Plot

In [66]:
#Select two predictor variables to visualize
boundary_predictors = ['BMICALC', 'ALCDAYSYR']

#Assign boundaries
X_boundary = X[boundary_predictors]
y_boundary = y

In [67]:
#Train Test Split
X_train_boundary, X_test_boundary, y_train_boundary, y_test_boundary = train_test_split(
    X_boundary, y_boundary, train_size=0.75, random_state=0, stratify=y_boundary
)

In [68]:
#Scale the predictors
boundary_scaler = StandardScaler()

X_train_boundary_scaled = boundary_scaler.fit_transform(X_train_boundary)
X_test_boundary_scaled = boundary_scaler.transform(X_test_boundary)

In [ ]:
#Fit simplified SVM
svm_boundary = SVC(kernel='linear', C=best_C_linear, random_state=0)

svm_boundary.fit(X_train_boundary_scaled, y_train_boundary)

In [ ]:
#Plot the decision boundary
DecisionBoundaryDisplay.from_estimator(
    svm_boundary,
    X_train_boundary_scaled,
    response_method='predict',
    xlabel='BMI, scaled',
    ylabel='Alcohol drinking days per year, scaled'
)

plt.scatter(
    X_train_boundary_scaled[:, 0],
    X_train_boundary_scaled[:, 1],
    c=y_train_boundary,
    edgecolors='k',
    s=20
)

plt.title('Simplified Linear SVM Decision Boundary')
plt.show()

The plot above shows that the model predicts nearly every observation as the same class of no heart disease (all the purple dots) and only a small portion of the observations are predicted to have heart disease (the yellow dots). The observations that are predicted to have heart disease are scattered in the plot and do not for a clearly separable cluster. This supports our earlier observations of high accuracy being misleading. The plot above shows the important limitation of how only these two predictors are not enough to create a clear boundary separating heart disease cases from non heart disease cases.

## 12. Discussion

In [ ]:
#Compare group means by target variable
nhis_model.groupby('CHEARTDIEV')[numeric_predictors].mean().round(2)

In [ ]:
#Compare group median values by target variable
nhis_model.groupby('CHEARTDIEV')[numeric_predictors].median().round(2)

### 12.1 Which predictors seemed most useful?
BMI seemed to be the most useful predictor in the Exploratory Data Analysis. Respondents with heart disease had a slightly higher average and median BMI than respondents without heart disease. The predictor 'MOD10DMIN' that quantifies moderate physical activity also had a small difference, with the respondents with heart disease having a lower median value of moderate physical activity. However, the difference between the groups were small overall. The predictors quantifying vegetable intake and sleep hours were nearly the same between the heart disease classification groupings. For the alcoholic beverages predictor, the means were similar while the medians differed between the two heart disease classification groups. Overall, these group differences being relatively small help explain why the SVM models had difficulty distinguishing heart disease cases from non heart disease cases.

### 12.2 Which model performed best?

Out of the three models (linear, radial, and polynomial SVMs), none of them reached the goal of being able to identify positive heart disease cases. Since the radial and polynomial SVMs (more complex models) did not improve the results compared to the linear SVM, the linear SVM model would be recommended out of the three. This performance difference was measured in recall, f1 score, and precision.

### 12.3 What does the model suggest about health habits or metrics?

Since the models did not have strong performance outcomes (based on their evaluation metrics of recall, f1 score, and precision), there is not any strong suggestions about health habits or metrics. The predictor variables of moderate physical activity, hours of sleep, sex, BMI, vegetable eating habits, and alcoholic beverages drank in a year may be related to heart disease. However, in this analysis using SVMs and this particular dataset, the predictors selected were not strong enough to separate the minority heart disease class from the majority non heart disease class.

### 12.4 Limitations of the Analysis
Because of the severe class imbalance in the target variable of heart disease, analysis was extremely limited.

The models in this analysis mostly predicted the majority class of no heart disease. Since a majority of the respondents in the dataset did not have heart disease, the accuracy rates were high, leading to misleading accuracy rates.

Only a small set of predictors was used.

Of the predictors used, many seem to be self reported, which allows for measurement errors (like weight of the respondent, which was used to calculate BMI).

This analysis is strictly observational, so causation cannot be supported.

### 12.5 Possible Improvements

Future modeling would have to address the class imbalance of the target variable (heart disease) more directly by using class weights, resampling, or tuning on another metric.

Trying additional predictors from the NHIS dataset may allow for more insight and clearer classifications.

Considering other models, like logistic regression or random forests, to compare to SVMs may also provide improvements.

Another possible method change would be to use adjusted thresholds to classify the observations. In practice, this would mean changing a default probability of greater than or equal to 50% being assigned the heart disease class to a user set threshold probability of, for example, greater than or equal to 35% probability. This could allow the model to identify more heart disease cases, increasing recall score for the heart disease class. However, lowering the threshold may increase false positives. This tradeoff would need to be balanced, which could be done by analyzing the evaluation metrics for various thresholds to find the optimal threshold. The other factor to consider with this suggestion is runtime, since calculating the probability of each observation is costly.

## 13. Poster Preparation Notes

### 13.1 Theory:

Support Vector Machines (SVMs) are models that classify observations based on a set of predictors. The decision boundary the SVM uses can be linear, radial, or polynomial. In all three of these SVM kernel types, the C parameter controls the margin flexibility of the classification boundary. Gamma, used in radial and polynomial kernel SVMs, controls the influence of the kernel. Degree, only used in polynomial kernel SVMs, controls the model's complexity.
In this analysis, the three SVM models were tuned using cross validation using F1-score. The goal when tuning an SVM is to minimize classification error while maximizing the margin.

### 13.2 Methodology
Target: heart disease indicator
Predictors: sex, moderate physical activity, sleep, BMI, vegetable intake, and alcoholic beverage frequency

To clean the data, observations with invalid or missing codes, including missing target variable value, were removed.

The predictor 'SEX' was one hot encoded since it was a binary variable.

The data was split into training and testing sets.

The numeric predictors were scaled using StandardScaler.

Linear, radial, and polynomial SVMs were fit.

Cross Validation was used to tune C (cost), gamma, and degree, when applicable, to each model.

Evaluation metrics were calculated for each best model for each kernel: accuracy, confusion matrix, precision, recall, and f1 score.

### 13.3 Key Results to Include

In [ ]:
#Model comparison table
model_comparison_table = pd.DataFrame({
    'Model': ['Linear SVM', 'Radial SVM', 'Polynomial SVM'],
    'Test Accuracy': [test_accuracy_linear, test_accuracy_radial, test_accuracy_poly],
    'Precision of Heart Disease Class': [precision_linear, precision_radial, precision_poly],
    'Recall of Heart Disease Class': [recall_linear, recall_radial, recall_poly],
    'F1 Score of Heart Disease Class': [f1_linear, f1_radial, f1_poly]
})

model_comparison_table.round(4)

### 13.4 Discussion
Accuracy was misleading because the target class was severly imbalanced. The models mostly predicted the majority class. Therefore, the models did not identify many cases of heart disease. Evaluation metrics like recall and F1-score for the heart disease class were poor. BMI showed the clearest small difference between target classification groups. The moderate physical activity predictor showed a smaller difference. The selected predictors, overall, were not strong enough to separate the heart disease cases. Future work should consider using class weights, resampling, more/different predictors, and threshold based evaluation.

This analysis is a great reminder to not evaluate health classification models using just accuracy. In rare health conditions, recall and false negatives are metrics to evaluate.

### 13.5 Figures to Export
Decision Boundary plot in section 11

### 13.6 Required Citation for Dataset

Blewett, L. A., Rivera Drew, J. A., King, M. L., Williams, K. C. W., Backman, D., Chen, A., & Richards, S. IPUMS Health Surveys: National Health Interview Survey, Version 7.4 [dataset]. Minneapolis, MN: IPUMS, 2024. https://doi.org/10.18128/D070.V7.4

### 13.7 Required Citations for documentation
Python, pandas, NumPy, matplotlib, scikit-learn

## 14. Conclusion

### 14.1 Main Takeaways
All three SVMs (linear, radial, and polynomial) had high accuracy, but that metric was misleading. A model can predict nearly all observations to be one class and still have a high accuracy rate, which was the case in this analysis.

The target variable (heart disease) had severe class imbalance. Trying to account for such with stratify=y and changing the scoring to be f1 was not sufficient in this case.

The models failed to identify heart disease cases, which was evident in the confusion matrices and in the positive class recall and f1 score results.

The selected predictors were not sufficient to identify heart disease cases in this dataset.

### 14.2 Policy or Real-World Implication

Since a diagnosis of certain health conditions are often rare, overall accuracy is not enough to evaluate performance of a model.

Thinking specifically in health diagnosis contexts, having more false positives (suggesting a negative health condition diagnosis needs to be explored) is more preferred than more false negatives (suggesting the respondent is healthy when there is actually a health condition needing to be addressed).

Minority class perfomance metrics are a must for health condition related classification models.

Models that are measured to have high accuracy may actually just be predicting that all respondents are in one target class, rendering the model useless.

### 14.3 Final Limitations

Class imbalance was a significant barrier in this analysis, rendering the models practically useless in predicting heart disease.

Only six predictors were used. More predictors or different predictors may have a stronger distinction between the target variable classes of having heart disease and not having heart disease.

How the predictors are measured (self reported) may pose problems in terms of legitimacy of the data itself.

This analysis is strictly observational, not causal. Therefore, health recommendations (for example, encouraging a specific amount of moderate physical exercise each week) are not able to be made from this analysis.

Support vector machines (in general and in this case analysis) may need imbalance handling, class weights, resamplig, or different predictors.

## 15. References

### 15.1 Dataset Citation
Lynn A. Blewett, Julia A. Rivera Drew, Miriam L. King, Kari C.W. Williams, Daniel Backman, Annie Chen, and Stephanie Richards. IPUMS Health Surveys: National Health Interview Survey, Version 7.4 [dataset]. Minneapolis, MN: IPUMS, 2024. https://doi.org/10.18128/D070.V7.4

### 15.2 Software/Package Citations
Python, pandas, NumPy, matplotlib, scikit-learn

### 15.3 Any external code or documentation referenced
scikit-learn SVC, GridSearchCV, StandardScaler, and classification metrics documentation